# Skylark Drones - Aerial GCP Pose Estimation (Google Colab GPU Training)

This notebook runs the **Two-Stage Coarse-to-Fine PyTorch pipeline** cleanly on Google Colab.

- **Google Drive mounting is optional:** If `drive.mount` fails (for example `MessageError: credential propagation was unsuccessful`), the notebook automatically downloads the dataset from the shared Google Drive link via `gdown` and saves all checkpoints/predictions locally under `/content/skylark-gcp-estimation/` for direct download.
- **Before running:** Select **Runtime -> Change runtime type -> T4 GPU** (or better), then run cells **1 through 12** in order.

In [ ]:
# 1. GPU Detection (Independent of Google Drive)
import subprocess
import torch

print('=== NVIDIA-SMI ===')
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(smi.stdout if smi.returncode == 0 else smi.stderr)

print('=== PyTorch CUDA Check ===')
print(f'PyTorch version: {torch.__version__}')
cuda_ok = torch.cuda.is_available()
print(f'CUDA available:  {cuda_ok}')

if cuda_ok:
    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    total_gb = props.total_memory / (1024 ** 3)
    free_gb, _ = torch.cuda.mem_get_info(device_idx)
    print(f'GPU Name:        {torch.cuda.get_device_name(device_idx)}')
    print(f'Total VRAM:      {total_gb:.2f} GB')
    print(f'Free VRAM:       {free_gb / (1024 ** 3):.2f} GB')
    _test = torch.ones((2, 2), device='cuda') * 2
    print(f'CUDA Tensor OK:  sum={_test.sum().item():.1f}')
else:
    print('WARNING: CUDA is NOT available! Switch Colab to Runtime -> Change runtime type -> T4 GPU.')

In [ ]:
# 2. Optional Google Drive Mounting (Non-Fatal)
DRIVE_MOUNTED = False
DRIVE_PROJECT = '/content/drive/MyDrive/skylark-gcp-estimation'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MOUNTED = True
    print(f'Google Drive mounted successfully. Backup path: {DRIVE_PROJECT}')
except Exception as exc:
    DRIVE_MOUNTED = False
    print('Google Drive mount failed (non-fatal — continuing without Drive mount).')
    print(f'Error details: {type(exc).__name__}: {exc}')
    print('Alternative workflow active:')
    print('  -> Cell 3 will download the dataset directly from the shared Drive link via gdown.')
    print('  -> Cells 7-11 will save weights & predictions under /content/skylark-gcp-estimation/ and zip them for download.')

In [ ]:
# 3. Dataset Download (Shared Google Drive Folder via gdown)
import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

LOCAL_PROJECT = Path('/content/skylark-gcp-estimation')
DATA_DIR = LOCAL_PROJECT / 'data'
TRAIN_DIR = DATA_DIR / 'train_dataset'
TEST_DIR = DATA_DIR / 'test_dataset'
SHARED_DRIVE_URL = 'https://drive.google.com/drive/folders/1RDNiAO1EynKrRDomcVNXQW1-ct5zzvE5?usp=sharing'

DATA_DIR.mkdir(parents=True, exist_ok=True)

def _count_images(root: Path) -> int:
    if not root.exists():
        return 0
    exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    return sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in exts)

def _find_json() -> Path | None:
    for cand in (TRAIN_DIR / 'gcp_marks.json', DATA_DIR / 'gcp_marks.json', TRAIN_DIR / 'curated_gcp_marks.json', DATA_DIR / 'curated_gcp_marks.json'):
        if cand.exists() and cand.stat().st_size > 0:
            return cand
    return None

# If Drive mounted and already contains data/, copy to fast local SSD first
drive_data = Path(DRIVE_PROJECT) / 'data'
if DRIVE_MOUNTED and drive_data.exists() and (_count_images(TRAIN_DIR) < 1000 or _count_images(TEST_DIR) < 300):
    print(f'Copying existing dataset from {drive_data} to {DATA_DIR} ...')
    shutil.copytree(drive_data, DATA_DIR, dirs_exist_ok=True)

existing_train = _count_images(TRAIN_DIR)
existing_test = _count_images(TEST_DIR)
existing_json = _find_json()

if existing_train == 1000 and existing_test == 300 and existing_json is not None:
    print(f'Dataset already complete: {existing_train} train, {existing_test} test, JSON={existing_json}')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'gdown>=5.0.0'])
    import gdown

    print('Listing shared Google Drive folder contents (may take 2-4 minutes for 1,301 files)...')
    try:
        drive_files = gdown.download_folder(
            url=SHARED_DRIVE_URL,
            output=str(DATA_DIR),
            skip_download=True,
            quiet=True,
        )
    except Exception as exc:
        raise RuntimeError(
            'Failed to list shared Google Drive folder. Check that the folder link is shared as '
            '"Anyone with the link (Viewer)" and try again. '
            f'Original error: {type(exc).__name__}: {exc}'
        ) from exc

    if not drive_files:
        raise RuntimeError(
            'Shared Google Drive folder returned 0 files. This usually means folder permissions are '
            'restricted or Google Drive blocked automated listing.'
        )

    print(f'Found {len(drive_files)} entries in shared Drive folder.')
    to_download = []
    for f in drive_files:
        fp = Path(f.path)
        out_path = fp if fp.is_absolute() else (DATA_DIR / fp)
        if not out_path.exists() or out_path.stat().st_size == 0:
            to_download.append((f.id, out_path))

    print(f'Need to download {len(to_download)} missing files (skipping {len(drive_files) - len(to_download)} existing).')

    def _download_one(item):
        fid, out_path = item
        out_path.parent.mkdir(parents=True, exist_ok=True)
        last_err = 'unknown error'
        for attempt in range(4):
            try:
                res = gdown.download(id=fid, output=str(out_path), quiet=True)
                if res and out_path.exists() and out_path.stat().st_size > 0:
                    return True, str(out_path), None
            except Exception as exc:
                last_err = f'{type(exc).__name__}: {exc}'
            time.sleep(1.5 * (attempt + 1))
        return False, str(out_path), last_err

    ok, failed = 0, []
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = [pool.submit(_download_one, item) for item in to_download]
        for idx, fut in enumerate(as_completed(futures), 1):
            success, p_str, err = fut.result()
            if success:
                ok += 1
            else:
                failed.append((p_str, err))
            if idx % 100 == 0 or idx == len(to_download):
                print(f'Download progress: {idx}/{len(to_download)} (ok={ok}, failed={len(failed)})')

    if failed:
        sample_errs = '\n'.join(f'  - {p}: {e}' for p, e in failed[:5])
        raise RuntimeError(
            f'{len(failed)} files failed to download (possibly Drive rate limit or permission issue). '
            'Re-run Cell 3 to resume missing files.\nSample errors:\n' + sample_errs
        )
    print(f'Download complete! Train images={_count_images(TRAIN_DIR)}, Test images={_count_images(TEST_DIR)}')

In [ ]:
# 4. Dataset Validation (Stops Before Training If Any Check Fails)
import json
import math
from collections import Counter
from pathlib import Path
from PIL import Image

LOCAL_PROJECT = Path('/content/skylark-gcp-estimation')
DATA_DIR = LOCAL_PROJECT / 'data'
TRAIN_DIR = DATA_DIR / 'train_dataset'
TEST_DIR = DATA_DIR / 'test_dataset'
ALLOWED_SHAPES = {'Cross', 'Square', 'L-Shape'}
IGNORE_INDEX = -1

# Auto-locate annotation JSON
ANNOTATIONS_PATH = None
for cand in (TRAIN_DIR / 'gcp_marks.json', DATA_DIR / 'gcp_marks.json', TRAIN_DIR / 'curated_gcp_marks.json', DATA_DIR / 'curated_gcp_marks.json'):
    if cand.exists():
        ANNOTATIONS_PATH = cand
        break
assert ANNOTATIONS_PATH is not None, f'Could not find gcp_marks.json in {TRAIN_DIR} or {DATA_DIR}'
print(f'Using annotations: {ANNOTATIONS_PATH}')

with open(ANNOTATIONS_PATH, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
train_files = sorted(p.relative_to(TRAIN_DIR).as_posix() for p in TRAIN_DIR.rglob('*') if p.is_file() and p.suffix.lower() in exts)
test_files = sorted(p.relative_to(TEST_DIR).as_posix() for p in TEST_DIR.rglob('*') if p.is_file() and p.suffix.lower() in exts)

assert len(train_files) == 1000, f'Expected 1000 train images, found {len(train_files)}'
assert len(test_files) == 300, f'Expected 300 test images, found {len(test_files)}'
assert len(annotations) == 1000, f'Expected 1000 annotation entries, found {len(annotations)}'

missing_paths = sorted(set(annotations.keys()) - set(train_files))
assert not missing_paths, f'{len(missing_paths)} annotation paths missing on disk: {missing_paths[:5]}'

train_sizes, test_sizes = Counter(), Counter()
corrupt_images = []
bad_coords = []
shape_counts = Counter()
ignored_count = 0

for rel in train_files:
    p = TRAIN_DIR / rel
    try:
        with Image.open(p) as im:
            w, h = im.size
        train_sizes[(w, h)] += 1
    except Exception as exc:
        corrupt_images.append((f'train/{rel}', str(exc)))
        continue

    entry = annotations[rel]
    x, y = entry['mark']['x'], entry['mark']['y']
    if not (isinstance(x, (int, float)) and isinstance(y, (int, float)) and math.isfinite(x) and math.isfinite(y) and 0 <= x < w and 0 <= y < h):
        bad_coords.append((rel, x, y, w, h))

    shape = entry.get('verified_shape')
    if shape is None:
        shape_counts[f'None (ignore_index={IGNORE_INDEX})'] += 1
        ignored_count += 1
    else:
        assert shape in ALLOWED_SHAPES, f'Invalid class label {shape!r} for {rel}'
        shape_counts[shape] += 1

for rel in test_files:
    p = TEST_DIR / rel
    try:
        with Image.open(p) as im:
            test_sizes[im.size] += 1
    except Exception as exc:
        corrupt_images.append((f'test/{rel}', str(exc)))

assert not corrupt_images, f'Found {len(corrupt_images)} corrupted images: {corrupt_images[:5]}'
assert not bad_coords, f'Found {len(bad_coords)} invalid/OOB coordinates: {bad_coords[:5]}'
assert ignored_count == 4, f'Expected 4 missing verified_shape entries, found {ignored_count}'

print('=== DATASET VALIDATION PASSED ===')
print(f'Train images: {len(train_files)} | Sizes: {dict(train_sizes)}')
print(f'Test images:  {len(test_files)}  | Sizes: {dict(test_sizes)}')
print(f'Annotations:  {len(annotations)} | Shapes: {dict(shape_counts)}')

In [ ]:
# 5. Dependency Installation
import subprocess
import sys
from pathlib import Path

req_file = Path('/content/skylark-gcp-estimation/requirements.txt')
if req_file.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)])
else:
    pkgs = [
        'torch>=2.1.0', 'torchvision>=0.16.0', 'numpy>=1.24.0', 'pandas>=2.0.0',
        'opencv-python-headless>=4.8.0', 'Pillow>=10.0.0', 'scikit-learn>=1.3.0',
        'matplotlib>=3.7.0', 'tqdm>=4.66.0', 'gdown>=5.0.0'
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

import cv2, matplotlib, numpy, pandas, sklearn, torch, torchvision, tqdm
print(f'Dependencies ready: torch={torch.__version__}, torchvision={torchvision.__version__}, cv2={cv2.__version__}, sklearn={sklearn.__version__}')

In [ ]:
# 6. Project Path Configuration (src/, weights/, outputs/, and working directory)
import os
import shutil
import sys
import zipfile
from pathlib import Path

LOCAL_PROJECT = Path('/content/skylark-gcp-estimation')
DATA_DIR = LOCAL_PROJECT / 'data'
TRAIN_DIR = DATA_DIR / 'train_dataset'
TEST_DIR = DATA_DIR / 'test_dataset'
WEIGHTS_DIR = LOCAL_PROJECT / 'weights'
OUTPUTS_DIR = LOCAL_PROJECT / 'outputs'

for d in (LOCAL_PROJECT / 'src', LOCAL_PROJECT / 'notebooks', WEIGHTS_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Optional: set your GitHub repo URL if Drive mount failed and you want Colab to pull src/ automatically
GITHUB_REPO_URL = ''

required_src = ['dataset.py', 'model.py', 'train.py', 'evaluate.py', 'predict.py']
missing_src = [f for f in required_src if not (LOCAL_PROJECT / 'src' / f).exists()]

if missing_src and DRIVE_MOUNTED and (Path(DRIVE_PROJECT) / 'src').exists():
    print(f'Copying source files from {DRIVE_PROJECT} ...')
    shutil.copytree(Path(DRIVE_PROJECT) / 'src', LOCAL_PROJECT / 'src', dirs_exist_ok=True)
    for extra in ('requirements.txt', 'README.md'):
        if (Path(DRIVE_PROJECT) / extra).exists():
            shutil.copy2(Path(DRIVE_PROJECT) / extra, LOCAL_PROJECT / extra)
    missing_src = [f for f in required_src if not (LOCAL_PROJECT / 'src' / f).exists()]

if missing_src and GITHUB_REPO_URL:
    tmp_clone = Path('/content/_repo_tmp')
    if tmp_clone.exists():
        shutil.rmtree(tmp_clone)
    import subprocess
    subprocess.check_call(['git', 'clone', '-q', GITHUB_REPO_URL, str(tmp_clone)])
    shutil.copytree(tmp_clone / 'src', LOCAL_PROJECT / 'src', dirs_exist_ok=True)
    missing_src = [f for f in required_src if not (LOCAL_PROJECT / 'src' / f).exists()]

if missing_src:
    print('Source files missing in /content/skylark-gcp-estimation/src/:', missing_src)
    print('Please upload a zip of your project code (src.zip) OR upload the src/*.py files now:')
    try:
        from google.colab import files
        uploaded = files.upload()
        for name, content in uploaded.items():
            if name.endswith('.zip'):
                zip_path = LOCAL_PROJECT / name
                zip_path.write_bytes(content)
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(LOCAL_PROJECT)
                # Flatten if zip contained top-level skylark-gcp-estimation/src
                nested_src = LOCAL_PROJECT / 'skylark-gcp-estimation' / 'src'
                if nested_src.exists():
                    shutil.copytree(nested_src, LOCAL_PROJECT / 'src', dirs_exist_ok=True)
            elif name.endswith('.py'):
                (LOCAL_PROJECT / 'src' / Path(name).name).write_bytes(content)
    except Exception as exc:
        print('Upload skipped or unavailable:', exc)
    missing_src = [f for f in required_src if not (LOCAL_PROJECT / 'src' / f).exists()]

assert not missing_src, (
    f'Missing required files in {LOCAL_PROJECT / "src"}: {missing_src}. '
    'Upload src/*.py (or src.zip) into /content/skylark-gcp-estimation/src/ and re-run Cell 6.'
)

# Auto-locate annotations JSON again cleanly
ANNOTATIONS_PATH = next(
    p for p in (TRAIN_DIR / 'gcp_marks.json', DATA_DIR / 'gcp_marks.json', TRAIN_DIR / 'curated_gcp_marks.json', DATA_DIR / 'curated_gcp_marks.json')
    if p.exists()
)

os.chdir(LOCAL_PROJECT)
if str(LOCAL_PROJECT) not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT))

from src.dataset import CLASS_NAMES, load_annotations, build_group_splits
from src.model import Stage1CoarseModel, Stage2FineModel

_samples = load_annotations(ANNOTATIONS_PATH)
_tr, _va = build_group_splits(_samples, n_splits=5, val_fold=3)
print(f'LOCAL_PROJECT    = {LOCAL_PROJECT}')
print(f'DATA_DIR         = {DATA_DIR}')
print(f'TRAIN_DIR        = {TRAIN_DIR}')
print(f'TEST_DIR         = {TEST_DIR}')
print(f'ANNOTATIONS_PATH = {ANNOTATIONS_PATH}')
print(f'Imports OK | Classes={CLASS_NAMES} | Fold-3 Split: train={len(_tr)}, val={len(_va)}')

In [ ]:
# 7. Stage 1 Training (Coarse ResNet-34 UNet Heatmap Localization at 1024x768)
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable, '-m', 'src.train',
    '--stage', '1',
    '--data-dir', str(TRAIN_DIR),
    '--annotations', str(ANNOTATIONS_PATH),
    '--weights-dir', str(WEIGHTS_DIR),
    '--outputs-dir', str(OUTPUTS_DIR),
    '--cache-dir', '/content/cache_stage1',
    '--val-fold', '3',
    '--n-splits', '5',
    '--epochs-s1', '35',
    '--batch-s1', '8',
    '--lr-s1', '3e-4',
    '--num-workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)
assert (WEIGHTS_DIR / 'stage1_best.pt').exists(), 'Stage 1 checkpoint missing!'

In [ ]:
# 8. Stage 2 Training (Fine 384x384 Native Crop Localization + 3-Class Shape Classification)
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'src.train',
    '--stage', '2',
    '--data-dir', str(TRAIN_DIR),
    '--annotations', str(ANNOTATIONS_PATH),
    '--weights-dir', str(WEIGHTS_DIR),
    '--outputs-dir', str(OUTPUTS_DIR),
    '--val-fold', '3',
    '--n-splits', '5',
    '--epochs-s2', '40',
    '--batch-s2', '16',
    '--lr-s2', '3e-4',
    '--num-workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)
assert (WEIGHTS_DIR / 'stage2_best.pt').exists(), 'Stage 2 checkpoint missing!'

In [ ]:
# 9. Validation Evaluation (End-to-End Two-Stage in Original Pixel Coordinates)
import subprocess
import sys
from IPython.display import Image, display

val_vis = OUTPUTS_DIR / 'val_predictions.png'
cmd = [
    sys.executable, '-m', 'src.evaluate',
    '--data-dir', str(TRAIN_DIR),
    '--annotations', str(ANNOTATIONS_PATH),
    '--stage1-weights', str(WEIGHTS_DIR / 'stage1_best.pt'),
    '--stage2-weights', str(WEIGHTS_DIR / 'stage2_best.pt'),
    '--val-fold', '3',
    '--n-splits', '5',
    '--vis-output', str(val_vis),
]
subprocess.check_call(cmd)
if val_vis.exists():
    display(Image(str(val_vis)))

In [ ]:
# 10. Test Inference (All 300 Test Images -> outputs/predictions.json)
import subprocess
import sys
from IPython.display import Image, display

pred_json = OUTPUTS_DIR / 'predictions.json'
test_vis = OUTPUTS_DIR / 'test_predictions_sample.png'
cmd = [
    sys.executable, '-m', 'src.predict',
    '--test-dir', str(TEST_DIR),
    '--stage1-weights', str(WEIGHTS_DIR / 'stage1_best.pt'),
    '--stage2-weights', str(WEIGHTS_DIR / 'stage2_best.pt'),
    '--output', str(pred_json),
    '--vis-output', str(test_vis),
]
subprocess.check_call(cmd)
if test_vis.exists():
    display(Image(str(test_vis)))

In [ ]:
# 11. Save Checkpoints and Predictions (Drive Backup if Mounted, Else Local + Download Zip)
import shutil
import zipfile
from pathlib import Path

bundle_zip = LOCAL_PROJECT / 'skylark_artifacts.zip'
with zipfile.ZipFile(bundle_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in (WEIGHTS_DIR, OUTPUTS_DIR):
        for p in sorted(folder.rglob('*')):
            if p.is_file():
                zf.write(p, arcname=p.relative_to(LOCAL_PROJECT).as_posix())
print(f'Packaged weights & outputs into: {bundle_zip} ({bundle_zip.stat().st_size / (1024**2):.2f} MB)')

if DRIVE_MOUNTED:
    drive_weights = Path(DRIVE_PROJECT) / 'weights'
    drive_outputs = Path(DRIVE_PROJECT) / 'outputs'
    shutil.copytree(WEIGHTS_DIR, drive_weights, dirs_exist_ok=True)
    shutil.copytree(OUTPUTS_DIR, drive_outputs, dirs_exist_ok=True)
    print(f'Backed up weights to: {drive_weights}')
    print(f'Backed up outputs to: {drive_outputs}')
else:
    print('Google Drive is NOT mounted. Downloading artifacts directly from Colab...')
    print('If browser blocks automatic download, use Colab left sidebar -> Files -> /content/skylark-gcp-estimation/skylark_artifacts.zip -> Download.')
    try:
        from google.colab import files
        files.download(str(OUTPUTS_DIR / 'predictions.json'))
        files.download(str(bundle_zip))
    except Exception as exc:
        print(f'Browser download trigger skipped ({exc}); download manually from Colab Files sidebar.')

In [ ]:
# 12. Final Output Verification
import json
import math
from collections import Counter
from pathlib import Path
from PIL import Image
import torch

from src.dataset import CLASS_NAMES
from src.model import Stage1CoarseModel, Stage2FineModel, load_checkpoint

# Verify checkpoints
s1_ckpt = WEIGHTS_DIR / 'stage1_best.pt'
s2_ckpt = WEIGHTS_DIR / 'stage2_best.pt'
assert s1_ckpt.exists() and s1_ckpt.stat().st_size > 0, f'Missing {s1_ckpt}'
assert s2_ckpt.exists() and s2_ckpt.stat().st_size > 0, f'Missing {s2_ckpt}'
load_checkpoint(s1_ckpt, Stage1CoarseModel(pretrained=False), map_location='cpu')
load_checkpoint(s2_ckpt, Stage2FineModel(pretrained=False), map_location='cpu')

# Verify predictions.json
pred_path = OUTPUTS_DIR / 'predictions.json'
assert pred_path.exists(), f'Missing {pred_path}'
preds = json.loads(pred_path.read_text(encoding='utf-8'))

exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
expected_keys = sorted(p.relative_to(TEST_DIR).as_posix() for p in TEST_DIR.rglob('*') if p.is_file() and p.suffix.lower() in exts)
assert len(expected_keys) == 300, f'Expected 300 test images, found {len(expected_keys)}'
assert set(preds.keys()) == set(expected_keys), 'Prediction keys do not match test_dataset image paths!'

shape_dist = Counter()
for k, v in preds.items():
    assert set(v.keys()) == {'mark', 'verified_shape'}, f'Invalid keys for {k}: {v.keys()}'
    x, y = v['mark']['x'], v['mark']['y']
    with Image.open(TEST_DIR / k) as im:
        w, h = im.size
    assert math.isfinite(x) and math.isfinite(y) and 0.0 <= x < w and 0.0 <= y < h, f'Invalid coords for {k}: ({x}, {y})'
    assert v['verified_shape'] in CLASS_NAMES, f'Invalid shape for {k}: {v["verified_shape"]}'
    shape_dist[v['verified_shape']] += 1

print('=== FINAL SUBMISSION VERIFICATION PASSED ===')
print(f'Stage 1 weights: {s1_ckpt} ({s1_ckpt.stat().st_size / (1024**2):.2f} MB)')
print(f'Stage 2 weights: {s2_ckpt} ({s2_ckpt.stat().st_size / (1024**2):.2f} MB)')
print(f'Predictions:     {pred_path} ({len(preds)} entries)')
print(f'Class counts:    {dict(shape_dist)}')